# TVI-DPGMM revision: 5 seeds

In [4]:
from Truncated_VI import vdpgmm
from sklearn import preprocessing
from sklearn import datasets
from sklearn.mixture import BayesianGaussianMixture
import matplotlib.pylab as plt
import matplotlib
import numpy as np
import scipy as sp
import torch
from scipy.optimize import linear_sum_assignment
from typing import List, Callable, Union, Any, TypeVar, Tuple
Tensor = TypeVar('torch.tensor')
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from collections import Counter
from scipy.special import logsumexp
import time

experiment_seeds = [42, 1, 2, 3, 6]

# Dataset

In [ ]:
'''use a numerical dataset to test the model
label  damaged_floor     damage_extent         number_of_samples
0           0            0%                     1500
1           1            5%                     500
2           1            10%                    500
3           2,4          10%, 10%               500
4           1,3,5        10%,15%,20%            500
5           2,4,6        15%,20%,25%            500
6           1,3,5,7      10%,15%,20%,25%        500
7           1,2,4,6,8    10%,15%,20%,25%,30%    500

'''
# Read data from CSV
features = pd.read_csv("TF_mag_numerical_8class_5000samples_20dB.csv")
features = features.astype("float32")
# # # Convert DataFrame to PyTorch tensors
X = torch.tensor(features.values[:7500,:])
X1 = X[1000:1500,:]
X2 = X[1800:2300,:]
X = torch.cat([X1, X2], dim=0)

X = X.t()

input_dim = X.shape[1]
print(X.shape)


a = []
for i in range(1500):
    a.append(0)
for i in np.arange(1500,2000):
    a.append(1)
for i in np.arange(2000,2500):
    a.append(2)
for i in np.arange(2500,3000):
    a.append(3)
for i in np.arange(3000,3500):
    a.append(4)
for i in np.arange(3500,4000):
    a.append(5)
for i in np.arange(4000,4500):
    a.append(6)
for i in np.arange(4500,5000):
    a.append(7)
print(len(a))
y0 = torch.tensor(a)
y0 = y0.unsqueeze(1)
y = [int(_) for _ in y0]
y = torch.tensor(y)
num_classes = y.max().item() + 1
print(f"number of total classes: {num_classes}")

plt.plot(y)
plt.show()

# Helper functions

In [6]:
def power_method(A, start=None, precision=1e-10):
    if start is None:
        start = torch.ones(len(A), 1)
    
    diff = precision + 1
    x = start
    n = torch.norm(x) + diff
    i = 0
    
    while diff > precision:
        i += 1
        y = torch.matmul(A, x)
        n2 = torch.norm(x)
        diff = abs(n2 - n)
        n = n2
        
        if n < 1.0e-200:
            x = torch.zeros(len(A), 1)
            break
        else:
            x = y / n
        
        if i > 100:
            break
    
    n = torch.norm(x)
    if n < 1.0e-200:
        vec = torch.zeros(len(A), 1)
    else:
        vec = x / n
    
    return vec, n

In [7]:
def brier_score(p_hat, y_true, classes=None, sample_weight=None):
    p_hat = np.asarray(p_hat, dtype=float)
    N, C = p_hat.shape
    if classes is None:
        classes = np.unique(y_true)
    class_to_idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    Y = np.eye(C)[y_idx]
    if sample_weight is None:
        return np.mean(np.sum((p_hat - Y)**2, axis=1))
    w = np.asarray(sample_weight, dtype=float)
    w /= w.sum()
    return np.sum(w * np.sum((p_hat - Y)**2, axis=1))

def brier_by_class(p_hat, y_true, classes=None):
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    class_to_idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([class_to_idx[y] for y in y_true], dtype=int)
    Y = np.eye(C)[y_idx]
    out = {}
    for c, cl in enumerate(classes):
        m = (y_idx == c)
        out[cl] = np.mean(np.sum((p_hat[m] - Y[m])**2, axis=1)) if m.any() else np.nan
    return out

def brier_skill_score(p_hat, y_true, classes=None):
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    counts = np.array([(y_true == c).sum() for c in classes], float)
    p_ref = counts / counts.sum()
    P_ref = np.tile(p_ref, (len(y_true), 1))
    BS = brier_score(p_hat, y_true, classes)
    BS_ref = brier_score(P_ref, y_true, classes)
    return 1.0 - (BS / BS_ref if BS_ref > 0 else np.nan)

def learn_Q_soft(R, y_true, classes=None, reg=1e-6):
    if classes is None:
        classes = np.unique(y_true)
    C, K = len(classes), R.shape[1]
    cls2idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([cls2idx[y] for y in y_true], int)
    E = np.zeros((C, K), float)
    for i in range(R.shape[0]):        
        E[y_idx[i]] += R[i]
    Q = (E + reg) / (E + reg).sum(axis=0, keepdims=True)
    return Q, classes

def class_probs_from_R(R, Q):
    return R @ Q.T  

def dpgmm_brier_from_responsibilities(
    R, y_true, labeled_mask=None, classes=None, sample_weight=None
):
    """
    Use the exact same pipeline as HDP-GMM: R -> learn Q on labeled subset -> P_hat -> Brier.
    """
    R = np.asarray(R, float)
    if labeled_mask is None:
        R_lab, y_lab = R, y_true
    else:
        m = np.asarray(labeled_mask, bool)
        R_lab, y_lab = R[m], np.asarray(y_true)[m]

    Q, classes = learn_Q_soft(R_lab, y_lab, classes=classes, reg=1e-6)
    P_hat = class_probs_from_R(R, Q)

    BS  = brier_score(P_hat, y_true, classes=classes, sample_weight=sample_weight)
    BSc = brier_by_class(P_hat, y_true, classes=classes)
    BSS = brier_skill_score(P_hat, y_true, classes=classes)
    return BS, BSc, BSS, P_hat, R, Q, classes

In [ ]:
def ece_toplabel(P_hat, y_true, n_bins=15):
    N, C = P_hat.shape
    conf = P_hat.max(axis=1)
    pred = P_hat.argmax(axis=1)
    correct = (pred == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx  = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = (idx == b)
        if not np.any(m): 
            continue
        acc_b  = correct[m].mean()
        conf_b = conf[m].mean()
        ece += m.mean() * abs(acc_b - conf_b)
    return ece

def ece_ovr(P_hat, y_true, n_bins=15, classes=None):
    P_hat = np.asarray(P_hat, float)
    N, C = P_hat.shape
    if classes is None:
        classes = np.arange(C)
    cls2idx = {c:i for i,c in enumerate(classes)}
    y_idx = np.array([cls2idx[y] for y in y_true], int)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for c in range(C):
        p_c = P_hat[:, c]
        idx = np.clip(np.digitize(p_c, bins) - 1, 0, n_bins - 1)
        for b in range(n_bins):
            m = (idx == b)
            if not np.any(m):
                continue
            acc_b  = (y_idx[m] == c).mean()
            conf_b = p_c[m].mean()
            ece += (m.sum() / N) * abs(acc_b - conf_b)
    return ece

def kfold_indices(N, k=5, shuffle=True, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(N)
    if shuffle:
        rng.shuffle(idx)
    return np.array_split(idx, k)

def probs_with_cv_Q(R, y_true, classes=None, kfold=5, reg=1e-6):
    """
    Learn Q on K-1 folds, predict P_hat on the held-out fold; repeat and stitch.
    R: (N,K) responsibilities from a fitted model (HDP or DPGMM)
    """
    N = R.shape[0]
    folds = kfold_indices(N, k=kfold, shuffle=True, seed=0)
    if classes is None:
        classes = np.unique(y_true)
    C = len(classes)
    P_hat = np.zeros((N, C), float)

    for val_idx in folds:
        train_idx = np.setdiff1d(np.arange(N), val_idx, assume_unique=True)
        Q, classes = learn_Q_soft(R[train_idx], y_true[train_idx], classes=classes, reg=reg)
        P_hat[val_idx] = class_probs_from_R(R[val_idx], Q)
    return P_hat, classes

def ece_ovr_classbalanced(P_hat, y_true, n_bins=15):
    N, C = P_hat.shape
    y_true = np.asarray(y_true)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    eces = []
    for c in range(C):
        p = P_hat[:, c]
        idx = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
        ece_c = 0.0
        for b in range(n_bins):
            m = (idx == b)
            if not m.any(): 
                continue
            acc_b  = (y_true[m] == c).mean()
            conf_b = p[m].mean()
            ece_c += (m.sum() / N) * abs(acc_b - conf_b)
        eces.append(ece_c)
    return float(np.mean(eces))


# Five-seed experiment

In [9]:
def unsupervised_clustering_accuracy(y: Union[np.ndarray, torch.Tensor], y_pred: Union[np.ndarray, torch.Tensor]) -> tuple:
    """Unsupervised Clustering Accuracy
    """
    assert len(y_pred) == len(y)
    u = np.unique(y)
    n_true_clusters = len(u)
    v = np.unique(y_pred)
    n_pred_clusters = len(v)
    map_u = dict(zip(u, range(n_true_clusters)))
    map_v = dict(zip(v, range(n_pred_clusters)))
    inv_map_u = {v: k for k, v in map_u.items()}
    inv_map_v = {v: k for k, v in map_v.items()}
    r = np.zeros((n_pred_clusters, n_true_clusters), dtype=np.int64)
    for y_pred_, y_ in zip(y_pred, y):
        if y_ in map_u:
            r[map_v[y_pred_], map_u[y_]] += 1
    reward_matrix  = np.concatenate((r, r, r), axis=1)
    cost_matrix = reward_matrix.max() - reward_matrix
    row_assign, col_assign = linear_sum_assignment(cost_matrix)

    # Construct optimal assignments matrix
    row_assign = row_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
    col_assign = col_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
    assignments = np.concatenate((row_assign, col_assign), axis=1)
    assignments = [[inv_map_v[x], inv_map_u[y%n_true_clusters]] for x, y in assignments]

    optimal_reward = reward_matrix[row_assign, col_assign].sum() * 1.0
    return optimal_reward / y_pred.size, assignments  

def damage_detection_accuracy(preds_new, healthy_count=1500, healthy_reference_fraction=0.8, min_count=10):
    health_labels_pre = np.unique(preds_new[:int(healthy_count * healthy_reference_fraction)])
    counts = Counter(preds_new[:int(healthy_count * healthy_reference_fraction)])
    health_labels = [label for label in health_labels_pre if counts[label] >= min_count]

    fn, fp = 0, 0
    for i in range(len(preds_new)):
        if i <= healthy_count and preds_new[i] not in health_labels:
            fp += 1
        elif i > healthy_count and preds_new[i] in health_labels:
            fn += 1

    dda = 1 - (fp + fn) / len(preds_new)
    return dda, fp, fn, health_labels


def run_tvi_dpgmm_once(seed, X, y):
    torch.manual_seed(seed)
    np.random.seed(seed)

    pca_components = 10
    samples_per_segment = 100

    pca = PCA(n_components=pca_components)
    data_pca = pca.fit_transform(X).astype("float32")
    print('PCA data shape:', data_pca.shape)

    data_gibbs = data_pca.reshape(-1, pca_components)
    data_gibbs_tensor = torch.tensor(data_gibbs)
    print('data_gibbs tensor shape:', data_gibbs_tensor.shape)

    dim = data_gibbs_tensor.shape[1]
    covariance0 = torch.cov(data_gibbs_tensor.T)
    diagonal_elements = torch.diag(covariance0)
    diagonal_covariance = torch.diag(diagonal_elements)
    covariance = diagonal_covariance

    if dim > 8:
        _, max_eig = power_method(covariance)
    else:
        max_eig = torch.max(torch.linalg.eig(covariance)[0].real)

    lambda_0 = 1
    nu_0 = dim + 2
    Psi_0 = nu_0 * max_eig * torch.eye(dim) * lambda_0
    Psi_0 = Psi_0.numpy()

    model = BayesianGaussianMixture(n_components = 20, tol=1e-300, max_iter = int(1e200), init_params = 'random', weight_concentration_prior_type='dirichlet_process',
                                 mean_precision_prior = lambda_0, degrees_of_freedom_prior = nu_0, covariance_prior = Psi_0, random_state=seed)

    # model = BayesianGaussianMixture(n_components=20,tol=1e-5,max_iter=int(1e5),init_params='kmeans',random_state=seed)

    data_gibbs = data_gibbs_tensor.numpy()

    start_time = time.time()
    model.fit(data_gibbs)
    runtime = time.time() - start_time

    preds = model.predict(data_gibbs)
    print('DPGMM')
    print(len(np.unique(preds)), np.unique(preds))
    print([np.sum(preds == label) for label in np.unique(preds)])

    preds_new = preds.copy()
    for new_label, old_label in enumerate(np.unique(preds)):
        preds_new[preds == old_label] = new_label

    y_true = y.numpy().squeeze().astype(int)

    acc, assignments = unsupervised_clustering_accuracy(y_true, preds_new.astype(int))
    dda, fp, fn, health_labels = damage_detection_accuracy(preds_new)

    resp = model.predict_proba(data_gibbs)

    BS, BSc, BSS, P_hat, R, Q, classes = dpgmm_brier_from_responsibilities(
        R=resp,
        y_true=y_true,
        labeled_mask=None,
        classes=None,
        sample_weight=None,
    )

    P_dpg, _ = probs_with_cv_Q(
        R=resp,
        y_true=y_true,
        classes=classes,
        kfold=5,
    )
    ece_dpg_top = ece_toplabel(P_dpg, y_true, n_bins=15)
    ece_dpg_ovr = ece_ovr(P_dpg, y_true=y_true, n_bins=15, classes=classes)
    ece_dpg_ovr_bal = ece_ovr_classbalanced(P_dpg, y_true, n_bins=15)

    return {
        'seed': seed,
        'accuracy': acc,
        'damage_detection_accuracy': dda,
        'false_positive': fp,
        'false_negative': fn,
        'brier_score': BS,
        'brier_skill_score': BSS,
        'ece_toplabel': ece_dpg_top,
        'ece_ovr': ece_dpg_ovr,
        'ece_balanced': ece_dpg_ovr_bal,
        'runtime_seconds': runtime,
        'num_components': len(np.unique(preds_new)),
        'assignments': assignments,
        'healthy_labels': health_labels,
        'predicted_clusters': preds_new.astype(int),
        'y_true': y_true,
    }

In [ ]:
seed_results = []

for run_id, seed in enumerate(experiment_seeds, start=1):
    print(f"\n===== Seed {seed} ({run_id}/{len(experiment_seeds)}) =====")
    result = run_tvi_dpgmm_once(seed=seed, X=X, y=y)
    seed_results.append(result)

    print(
        f"Seed {seed}: ACC={result['accuracy']:.6f}, "
        f"DDA={result['damage_detection_accuracy']:.6f}, "
        f"Brier={result['brier_score']:.6f}, "
        f"BSS={result['brier_skill_score']:.6f}, "
        f"ECE_top={result['ece_toplabel']:.6f}, "
        f"runtime={result['runtime_seconds']:.2f}s, "
        f"K={result['num_components']}"
    )

results_df = pd.DataFrame(seed_results)

metric_columns = [
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
]

summary_df = pd.DataFrame({
    'mean': results_df[metric_columns].mean(),
    'std': results_df[metric_columns].std(ddof=1),
})

In [ ]:
print('\nPer-seed results')
display(results_df[[
    'seed',
    'accuracy',
    'damage_detection_accuracy',
    'false_positive',
    'false_negative',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
    'num_components',
]])

print('\nMean and sample standard deviation over seeds')
display(summary_df)